# 06 - SASRec (Self-Attentive Sequential Recommendation)

SASRec uses **causal self-attention** over the city sequence: at each step it learns which earlier stops matter most for predicting the next, rather than compressing everything into one state (GRU4Rec) or reading only the last city (Markov / FPMC). It is the natural place to add side features.

For the benchmark it is trained with **full-catalogue cross-entropy** and **AdamW**, the same objective and optimiser as GRU4Rec, so the GRU4Rec-vs-SASRec comparison isolates recurrence vs attention (the original paper's BPR loss is still available via `loss="bpr"`).

### Benchmark protocol (same as GRU4Rec / Markov)
1. **Tune** with seed 42: train on Train, select on validation **NDCG@10** -- 25 Optuna TPE trials (median pruner), full-catalogue cross-entropy, AdamW.
2. **Record $E_{best}$**, the peak epoch of the winning trial.
3. **Merge** Train + Validation.
4. **Train 5 fresh models** (seeds `[1, 7, 13, 2024, 99]`) for exactly $E_{best}$ epochs, no early stopping.
5. **Score on Test**; log **mean** and **std** across seeds to `results.csv`.

**Search space** (the unified space): `embed_dim` {64,128,256}, `num_blocks` {1,2}, `dropout` [0,0.5], `lr` [1e-4,1e-2] log, `weight_decay` [1e-6,1e-3] log, `batch_size` {64,128,256}, `max_len` {10,20,30,40} (same values as GRU4Rec; trips are short, so longer windows are identical). `n_heads` is fixed at 2.

This is done three times -- **no side features**, side features **summed**, side features **concatenated** -- each with its own search and the identical budget.

All logic is in `src/models/sasrec.py` (a hand-rolled transformer so the causal + padding masks are fully controlled). Running this notebook top to bottom does everything and ends by displaying `results.csv`.

> **Runtime:** 3 variants x 25 trials, then 3 x 5 final fits. Use a GPU. To smoke-test the pipeline first, set `N_TRIALS = 2` and `SEEDS = SEEDS[:2]` in the config cell below -- and don't trust those numbers.

In [1]:
import torch

import os, sys, json
def find_root(start=None):
    d = os.path.abspath(start or os.getcwd())
    for _ in range(6):
        if os.path.exists(os.path.join(d, 'src', 'preprocessing.py')):
            return d
        d = os.path.dirname(d)
    return os.path.abspath(os.path.join(os.getcwd(), '..'))
ROOT = find_root(); sys.path.insert(0, ROOT)
import pandas as pd
from src.preprocessing import load_processed, preprocess
from src.evaluate import append_result
try:
    data = load_processed(os.path.join(ROOT, 'data', 'processed'))
except FileNotFoundError:
    data = preprocess(data_dir=os.path.join(ROOT, 'data'), sample_n_trips=5000)
print('loaded:', data['meta'])
if data['meta'].get('sample_n_trips') is not None:
    print('WARNING: processed.pkl was built on a subsample (sample_n_trips=%r). '
          'Delete data/processed/processed.pkl and rebuild with sample_n_trips=None '
          'before trusting or logging any numbers.' % data['meta']['sample_n_trips'])

RESULTS = os.path.join(ROOT, 'results.csv')
TUNING_DIR = os.path.join(ROOT, 'results', 'tuning'); os.makedirs(TUNING_DIR, exist_ok=True)

if torch.cuda.is_available():
    DEVICE = 'cuda'
elif torch.backends.mps.is_available():          # Apple Silicon GPU
    DEVICE = 'mps'
else:
    DEVICE = 'cpu'
print('device:', DEVICE)

loaded: {'min_trip_len': 4, 'val_frac': 0.1, 'sample_n_trips': None, 'seed': 42, 'n_train_trips': 190737, 'n_valid_trips': 21187, 'n_test_trips': 70662, 'n_cities': 39903, 'n_users': 200153, 'n_personas': 15, 'n_test_targets_unseen': 0}
device: cuda


In [2]:
from src.models.sasrec import tune_sasrec as TUNE, final_sasrec as FINAL, SIDE_FEATURES
from src.models.gru4rec import N_TRIALS, FINAL_SEEDS, MAX_EPOCHS, PATIENCE
SEEDS = list(FINAL_SEEDS)
# N_TRIALS = 2; SEEDS = SEEDS[:2]        # <- uncomment for a quick smoke test only
print(f"trials/variant={N_TRIALS} | max_epochs={MAX_EPOCHS} | patience={PATIENCE} | seeds={SEEDS}")

trials/variant=25 | max_epochs=20 | patience=3 | seeds=[1, 7, 13, 2024, 99]


## Ablation - side features on/off, and how they combine

We compare (a) no side features, (b) side features **summed** into the city embedding, and (c) side features **concatenated** then projected back to the model width. The features are trip-level context: `device_class`, `booker_country`, `affiliate_id`. This isolates whether context moves the metrics and whether sum or concat is better.

For each variant: **tune** (validation only) -> read $E_{best}$ -> **final fits** on Train + Val for 5 seeds -> append the mean row and the std row to `results.csv`, and save the per-seed table and chosen hyper-parameters under `results/tuning/`.

Rows written: `SASRec-no-side`, `SASRec-side-sum`, `SASRec-side-concat`, each with a `(std)` row.

In [3]:
def run_variant(tag, row_name, tune_kw):
    """Steps 1-5 for one variant: tune on validation, read E_best, 5-seed final
    fits on train+val, save params + per-seed table, log mean and std rows."""
    print(f"\n==================== {row_name} ====================")

    # steps 1-2: search on validation, record E_best
    study = TUNE(data, n_trials=N_TRIALS, device=DEVICE, show_progress_bar=False, **tune_kw)
    best_params = dict(study.best_params)
    e_best = study.best_trial.user_attrs["best_epoch"]
    with open(os.path.join(TUNING_DIR, f"{tag}.json"), "w") as f:
        json.dump({"best_params": best_params, "E_best": e_best,
                   "best_val_NDCG@10": study.best_value, "n_trials": len(study.trials)}, f, indent=2)

    # steps 3-5: train+val, 5 seeds x E_best epochs, score on test
    per_seed, _ = FINAL(data, best_params, epochs=e_best, seeds=SEEDS, device=DEVICE, **tune_kw)
    per_seed.to_csv(os.path.join(TUNING_DIR, f"{tag}-per-seed.csv"))

    mean, std = per_seed.mean(), per_seed.std()
    mean_row = {k: (int(v) if k == "n_params" else float(v)) for k, v in mean.items()}
    std_row = {k: float(v) for k, v in std.items()}
    append_result(RESULTS, row_name, mean_row)
    append_result(RESULTS, f"{row_name} (std)", std_row)
    summaries[row_name] = {"best_params": best_params, "E_best": e_best,
                           "val_NDCG@10": round(study.best_value, 4),
                           "test_NDCG@10": f"{mean['NDCG@10']:.4f} +/- {std['NDCG@10']:.4f}"}

summaries = {}

### No side features

In [4]:
run_variant("sasrec-no-side", "SASRec-no-side", {"side_feats": ()})


==================== SASRec-no-side ====================


C:\Users\itzam\AppData\Local\Temp\claude\c--Users-itzam-OneDrive-Desktop-SC4020-SC4020-PROJECT1\2c2033c1-ba24-48f1-8b0d-e2dba6a17bfe\scratchpad\gpuenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


SASRec (no side): best val NDCG@10 = 0.4652 (10/25 completed, 15 pruned, 13020s)
  params: {'lr': 0.0002981158737685572, 'weight_decay': 5.2209890824242905e-05, 'dropout': 0.11501455023567547, 'embed_dim': 256, 'num_blocks': 2, 'batch_size': 256, 'max_len': 30}
  best epoch (E_best): 5



SASRec (no side) - 5 seeds
                    mean      std
HR@1        2.769000e-01   0.0006
NDCG@1      2.769000e-01   0.0006
Accuracy@4  4.976000e-01   0.0014
HR@5        5.360000e-01   0.0017
NDCG@5      4.130000e-01   0.0009
HR@10       6.522000e-01   0.0021
NDCG@10     4.507000e-01   0.0010
MRR         3.879000e-01   0.0007
n_trips     7.066200e+04   0.0000
train_secs  5.905600e+02  38.3254
n_params    1.180288e+07   0.0000


### Side features - sum

In [5]:
run_variant("sasrec-side-sum", "SASRec-side-sum", {"side_feats": SIDE_FEATURES, "side_mode": "sum"})


==================== SASRec-side-sum ====================


SASRec (sum): best val NDCG@10 = 0.4693 (10/25 completed, 15 pruned, 10838s)
  params: {'lr': 0.0002981158737685572, 'weight_decay': 5.2209890824242905e-05, 'dropout': 0.11501455023567547, 'embed_dim': 256, 'num_blocks': 2, 'batch_size': 256, 'max_len': 30}
  best epoch (E_best): 6



SASRec (sum) - 5 seeds
                    mean      std
HR@1        2.811000e-01   0.0010
NDCG@1      2.811000e-01   0.0010
Accuracy@4  5.003000e-01   0.0025
HR@5        5.380000e-01   0.0020
NDCG@5      4.159000e-01   0.0005
HR@10       6.546000e-01   0.0025
NDCG@10     4.537000e-01   0.0007
MRR         3.912000e-01   0.0003
n_trips     7.066200e+04   0.0000
train_secs  6.705600e+02  21.0472
n_params    1.263949e+07   0.0000


### Side features - concat

In [6]:
run_variant("sasrec-side-concat", "SASRec-side-concat", {"side_feats": SIDE_FEATURES, "side_mode": "concat"})


==================== SASRec-side-concat ====================


SASRec (concat): best val NDCG@10 = 0.4730 (12/25 completed, 13 pruned, 11841s)
  params: {'lr': 0.0002981158737685572, 'weight_decay': 5.2209890824242905e-05, 'dropout': 0.11501455023567547, 'embed_dim': 256, 'num_blocks': 2, 'batch_size': 256, 'max_len': 30}
  best epoch (E_best): 5



SASRec (concat) - 5 seeds
                    mean     std
HR@1        2.794000e-01  0.0009
NDCG@1      2.794000e-01  0.0009
Accuracy@4  4.859000e-01  0.0035
HR@5        5.213000e-01  0.0031
NDCG@5      4.066000e-01  0.0019
HR@10       6.345000e-01  0.0034
NDCG@10     4.432000e-01  0.0019
MRR         3.837000e-01  0.0015
n_trips     7.066200e+04  0.0000
train_secs  6.365400e+02  7.4319
n_params    1.238528e+07  0.0000


## Selected configurations

In [7]:
display(pd.DataFrame(summaries).T)

,best_params,E_best,val_NDCG@10,test_NDCG@10
SASRec-no-side,"{'lr': 0.0002981158737685572, 'weight_decay': ...",5,0.4652,0.4507 +/- 0.0010
SASRec-side-sum,"{'lr': 0.0002981158737685572, 'weight_decay': ...",6,0.4693,0.4537 +/- 0.0007
SASRec-side-concat,"{'lr': 0.0002981158737685572, 'weight_decay': ...",5,0.473,0.4432 +/- 0.0019


## Final results (`results.csv`)

In [8]:
display(pd.read_csv(RESULTS))

,model,HR@1,HR@5,HR@10,NDCG@5,NDCG@10,MRR,Accuracy@4,n_trips
0,Markov,0.233464,0.444227,0.521284,0.345555,0.370604,0.323141,0.417056,70662.0
1,Markov (no recursion),0.198056,0.430585,0.517945,0.321054,0.349452,0.296480,0.398630,70662.0
2,GRU4Rec-no-side,0.252393,0.516023,0.639147,0.389637,0.429617,0.364470,0.474357,70662.0
3,GRU4Rec-side,0.258193,0.528782,0.645977,0.400009,0.438064,0.373161,0.488732,70662.0
4,SASRec-no-side,0.276921,0.536034,0.652212,0.413024,0.450680,0.387860,0.497620,70662.0
5,SASRec-no-side (std),0.000584,0.001729,0.002063,0.000853,0.000982,0.000747,0.001399,0.0
6,SASRec-side-sum,0.281130,0.538018,0.654598,0.415934,0.453741,0.391156,0.500328,70662.0
7,SASRec-side-sum (std),0.001022,0.001981,0.002483,0.000544,0.000698,0.000285,0.002543,0.0
8,SASRec-side-concat,0.279384,0.521262,0.634542,0.406590,0.443247,0.383730,0.485910,70662.0
9,SASRec-side-concat (std),0.000859,0.003102,0.003434,0.001873,0.001887,0.001490,0.003546,0.0


## What to write up

- Report the side-feature ablation (none vs sum vs concat) as mean +/- std over 5 seeds, next to GRU4Rec's side / no-side pair. A gap smaller than the std is not a real effect; if context doesn't help either neural model, that is a finding about the dataset.
- GRU4Rec vs SASRec now share loss, optimiser, search space and budget, so their gap is recurrence vs attention.
- Contrast SASRec with FPMC/Markov by **trip length** (the comparison notebook slices this): the attention advantage should show up on longer trips.
- Compute: see `n_params` / `train_secs` in the per-seed CSVs for the accuracy-vs-cost discussion.